Training a model is only half the job: to be useful, other programs have to be able to ask it for predictions. That step is called **serving**. This nb walks through it with `mlflow`, using a small XGBoost regressor on the California housing dataset (predict a district's median house value from 8 census features).

We build up in three layers:
1. **Package** the model so it is self-contained and describes its own inputs and outputs.
2. **Register** it under a stable name, so consumers ask for `california-housing-xgb` rather than for one particular run.
3. **Serve** it: first by loading it in-process, then as a REST API that any language can call.

By the end we will have called a live prediction endpoint and swapped the model behind it without changing the calling code.

## Set Up `mlflow`

In [ ]:
import mlflow

# point the client at the local mlflow server
# if you don't do this, mlflow will log to a local directory called mlruns
mlflow.set_tracking_uri("http://127.0.0.1:5000")

# set a new experiment
mlflow.set_experiment("mlflow-model-serving")

# inspect the experiment
experiment = mlflow.get_experiment_by_name("mlflow-model-serving")
print(f"Experiment_id: {experiment.experiment_id}")
print(f"Artifact Location: {experiment.artifact_location}")
print(f"Tags: {experiment.tags}")
print(f"Lifecycle_stage: {experiment.lifecycle_stage}")

## Prep Data

The California housing dataset has 20,640 census districts, 8 numeric features (median income, house age, average rooms, latitude, ...), and a target: the district's median house value in units of $100,000. It ships with `scikit-learn` and downloads once, then is cached locally.

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

X, y = fetch_california_housing(as_frame=True, return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
X_train.head(3)

## Train and package a model

Logging a model with `mlflow.xgboost.log_model` does more than save the weights. It writes a directory in the **MLflow Model format**: the model file plus metadata describing how to load it and what it expects. The pieces that matter for serving:

- **Flavor**: how to load the model (here, `xgboost`). Every model also gets a generic `python_function` (**pyfunc**) flavor, which gives any MLflow model the same `predict(DataFrame)` interface. Serving tools only need pyfunc, so they work the same for any framework.
- **Signature**: the names and types of the input columns and of the output. It is enforced at predict time.
- **Environment files**: the Python and package versions the model was trained with, so a server can recreate a compatible environment.
- **Input example**: a few real rows, also saved as a ready-to-send request body.

In [ ]:
from __future__ import annotations

from mlflow.models import infer_signature
from mlflow.models.model import ModelInfo
from sklearn.metrics import root_mean_squared_error
from xgboost import XGBRegressor


def train_and_log_model(**xgb_params) -> tuple[ModelInfo, float]:
    """Train an XGBRegressor and log it to a new run, returning the model info and test RMSE."""
    with mlflow.start_run():
        model = XGBRegressor(random_state=42, **xgb_params).fit(X_train, y_train)
        rmse = root_mean_squared_error(y_test, model.predict(X_test))
        mlflow.log_params(xgb_params)
        mlflow.log_metric("rmse", rmse)

        # the signature is inferred from sample inputs and outputs; the input example
        # is a few real rows stored with the model (and as a ready-to-send request body)
        model_info = mlflow.xgboost.log_model(
            model,
            name="model",
            signature=infer_signature(X_train, model.predict(X_train)),
            input_example=X_train.head(3),
        )
    return model_info, rmse


model_info_v1, rmse_v1 = train_and_log_model(n_estimators=100, max_depth=4)
print(f"Logged model: {model_info_v1.model_uri} | test RMSE: {rmse_v1:.4f}")

## Register the model

So far the model is addressable only by a run or model ID, which changes every time we retrain. The **Model Registry** gives it a stable name with numbered **versions**, plus **aliases**: movable labels, like `champion`, that point at one version.

Consumers then load `models:/california-housing-xgb@champion`. When a better version appears we move the alias, and nothing on the consumer side changes. The registry lives in the same backend database as the runs, and you can browse it in the UI's **Models** tab.

In [ ]:
MODEL_NAME = "california-housing-xgb"
ALIAS = "champion"
MODEL_URI = f"models:/{MODEL_NAME}@{ALIAS}"

client = mlflow.MlflowClient()

# register_model creates the named model on first use, then adds a numbered version
version_1 = mlflow.register_model(model_info_v1.model_uri, MODEL_NAME)

# point the alias at version 1
client.set_registered_model_alias(MODEL_NAME, ALIAS, version_1.version)

champion = client.get_model_version_by_alias(MODEL_NAME, ALIAS)
print(f"{MODEL_URI} -> version {champion.version}")

### What got saved

The cell below downloads the registered version's directory and prints its `MLmodel` file, which is the model's table of contents. Look for the `python_function` flavor (what serving tools use), the `xgboost` flavor, and the `signature` block listing every input column and its type.

| File | Purpose |
|---|---|
| `MLmodel` | Metadata: flavors, signature, MLflow version, input example info |
| `model.ubj` | The trained XGBoost model |
| `conda.yaml`, `python_env.yaml`, `requirements.txt` | The environment the model needs |
| `input_example.json`, `serving_input_example.json` | Sample rows, and the same rows as a request body |

In [ ]:
from pathlib import Path

# download the whole model directory from the registry to a local temp folder
model_dir = Path(mlflow.artifacts.download_artifacts(artifact_uri=f"models:/{MODEL_NAME}/{version_1.version}"))

print("Files:", sorted(p.name for p in model_dir.iterdir()))
print()
print((model_dir / "MLmodel").read_text())

## Serving, level 1: load it in-process

The simplest form of serving is loading the model inside the program that needs predictions (a batch job, a notebook, an app). `mlflow.pyfunc.load_model` returns the same `predict` interface regardless of the framework that trained the model.

It also enforces the signature, so malformed input fails with a clear message instead of a silent wrong answer.

In [ ]:
model = mlflow.pyfunc.load_model(MODEL_URI)  # resolved through the registry alias

batch = X_test.head(3)
local_preds = model.predict(batch)
print("Predictions:", local_preds)
print("Actual:     ", y_test.head(3).to_numpy())

# the signature rejects input that doesn't match the schema it was trained on;
# the reason is at the end of the exception message, after the echoed input data
try:
    model.predict(batch.drop(columns=["AveRooms"]))
except Exception as e:
    print(f"\nMissing column -> {type(e).__name__}: {str(e).split('Error: ')[-1]}")

## Serving, level 2: a REST API

In-process loading requires the caller to be Python with `mlflow` and `xgboost` installed. A model **server** removes that requirement: it loads the model once and exposes it over HTTP, so any language or system can ask for predictions with a JSON request.

`mlflow models serve` does this with one command. We launch it as a background process from the notebook:
- `-m` is the model to serve. We pass the alias URI, which the server resolves through the registry **once, at startup**.
- `--env-manager local` runs the model in the current environment. By default MLflow builds a fresh environment from the model's saved requirements, which is safer for production but slow for a demo.

The server exposes `/ping` and `/health` (liveness checks), `/version`, and `/invocations` (predictions). The helpers below start the process, wait until it answers, and shut it down with all its worker processes.

In [ ]:
import os
import subprocess
import sys
import tempfile
import time

import numpy as np
import pandas as pd
import psutil
import requests

PORT = 5001  # the tracking server already uses 5000
BASE_URL = f"http://127.0.0.1:{PORT}"
SERVE_LOG = Path(tempfile.mkdtemp()) / "model_server.log"


def stop_server(proc: subprocess.Popen) -> None:
    """Kill the model server and every worker process it spawned."""
    try:
        parent = psutil.Process(proc.pid)
        procs = [*parent.children(recursive=True), parent]
    except psutil.NoSuchProcess:
        return
    for p in procs:
        try:
            p.kill()
        except psutil.NoSuchProcess:
            pass
    proc.wait()


def start_server(model_uri: str, timeout: int = 120) -> subprocess.Popen:
    """Launch `mlflow models serve` in the background and block until it answers /ping."""
    # the server needs the tracking URI to resolve models:/ URIs through the registry
    env = {**os.environ, "MLFLOW_TRACKING_URI": mlflow.get_tracking_uri()}
    cmd = [sys.executable, "-m", "mlflow", "models", "serve", "-m", model_uri, "-p", str(PORT)]
    with open(SERVE_LOG, "w") as log:
        proc = subprocess.Popen(cmd + ["--env-manager", "local"], env=env, stdout=log, stderr=subprocess.STDOUT)

    deadline = time.time() + timeout
    while time.time() < deadline:
        if proc.poll() is not None:
            raise RuntimeError(f"model server exited early:\n{SERVE_LOG.read_text()[-1000:]}")
        try:
            if requests.get(f"{BASE_URL}/ping", timeout=1).status_code == 200:
                return proc
        except requests.ConnectionError:
            pass
        time.sleep(1)

    stop_server(proc)
    raise TimeoutError(f"model server did not start within {timeout}s; see {SERVE_LOG}")


server = start_server(MODEL_URI)
print(f"Serving {MODEL_URI} at {BASE_URL}")

### Call the endpoint

`/invocations` accepts JSON in a few shapes. `dataframe_split` (column names plus rows) and `dataframe_records` (a list of row dicts) both work for a model whose signature names its columns. The response is always `{"predictions": [...]}`.

The second half of the cell sends a request with the wrong columns. The server returns a `400` and an error message instead of a prediction, which is the same signature check we saw in-process, now enforced at the network boundary.

In [ ]:
def predict_rest(df: pd.DataFrame) -> np.ndarray:
    """POST a DataFrame to the model server and return its predictions."""
    payload = {"dataframe_split": {"columns": list(df.columns), "data": df.values.tolist()}}
    response = requests.post(f"{BASE_URL}/invocations", json=payload, timeout=30)
    response.raise_for_status()
    return np.array(response.json()["predictions"])


# liveness and metadata endpoints
for path in ["/ping", "/health", "/version"]:
    r = requests.get(BASE_URL + path)
    print(f"GET  {path:<9} -> {r.status_code} {r.text!r}")

# two accepted request shapes for the same rows
payloads = {
    "dataframe_split": {"dataframe_split": {"columns": list(batch.columns), "data": batch.values.tolist()}},
    "dataframe_records": {"dataframe_records": batch.to_dict(orient="records")},
}
for label, payload in payloads.items():
    r = requests.post(f"{BASE_URL}/invocations", json=payload)
    print(f"POST /invocations ({label}) -> {r.status_code} {r.json()}")

# the REST answer should match the in-process answer
print("\nMatches in-process predictions:", np.allclose(predict_rest(batch), local_preds))

# a request that violates the signature is rejected
bad = requests.post(f"{BASE_URL}/invocations", json={"dataframe_split": {"columns": ["MedInc"], "data": [[3.0]]}})
print(f"Wrong columns -> {bad.status_code} {bad.json()['error_code']}")

# rough per-request cost for a 3-row batch, measured on this machine
start = time.perf_counter()
for _ in range(50):
    predict_rest(batch)
print(f"~{(time.perf_counter() - start) / 50 * 1000:.1f} ms per request")

## Promote a new version without touching clients

The payoff of registering and aliasing: we can replace the model behind the endpoint without changing any calling code. We train a bigger model, register it as version 2, and move the `champion` alias to it.

Because the server resolved the alias when it started, it keeps serving version 1 until it is restarted. Compare what the live server returns before and after the restart with the new champion's predictions. The calling code (`predict_rest`) is identical both times.

In [ ]:
# train and register a larger model as version 2
model_info_v2, rmse_v2 = train_and_log_model(n_estimators=400, max_depth=6, learning_rate=0.05)
version_2 = mlflow.register_model(model_info_v2.model_uri, MODEL_NAME)
print(f"RMSE: v1 {rmse_v1:.4f} -> v2 {rmse_v2:.4f}")

# promote: move the alias, so models:/...@champion now means version 2
client.set_registered_model_alias(MODEL_NAME, ALIAS, version_2.version)
print(f"{MODEL_URI} -> version {client.get_model_version_by_alias(MODEL_NAME, ALIAS).version}")

# what a fresh load of the alias returns now
new_champion_preds = mlflow.pyfunc.load_model(MODEL_URI).predict(batch)

# the running server still holds version 1
print("\nBefore restart, server matches new champion:", np.allclose(predict_rest(batch), new_champion_preds))

# restart: the server re-resolves the alias and loads version 2
stop_server(server)
server = start_server(MODEL_URI)
print("After restart,  server matches new champion:", np.allclose(predict_rest(batch), new_champion_preds))

## Shut down

The server is a separate process that keeps running (and holding port 5001) until we stop it. Always stop it when you're done, or it will outlive the notebook.

In [ ]:
stop_server(server)

try:
    requests.get(f"{BASE_URL}/ping", timeout=2)
    print("Server is still responding")
except requests.ConnectionError:
    print("Server stopped")

## Where to go from here

What we ran is a development setup: one local process, no authentication, no scaling. The same packaged model carries over to heavier options, because everything the server needs (model, signature, environment) is already in the MLflow Model directory.

| Need | Option |
|---|---|
| Check the model works in its own freshly built environment before deploying | `mlflow.models.predict(model_uri, input_data, env_manager="uv")` |
| Ship it as a container | `mlflow models build-docker` |
| Score a large dataset in batch | `mlflow.pyfunc.spark_udf` (Spark) |
| Deploy to a managed platform | MLflow's deployment integrations for cloud ML services |

The takeaway is the division of labor: training code produces a self-describing model, the registry gives it a stable name and lifecycle, and serving is a thin layer that only needs the name.